In [1]:
import os
import re
import faiss
import numpy as np
from openai import OpenAI

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [2]:
# 1) Load document
with open("History_of_Art.txt", "r", encoding="utf-8") as f:
    text = f.read()

In [9]:
paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]

chunks = []
current = ""

for p in paragraphs:
    if len(current) + len(p) < 500:
        current += "\n\n" + p if current else p
    else:
        chunks.append(current)
        current = p

if current:
    chunks.append(current)


In [10]:
# 3) Embed chunks
embedding_response = client.embeddings.create(
    model="text-embedding-3-small",
    input=chunks
)
chunk_vectors = [item.embedding for item in embedding_response.data]

In [ ]:
# 4) Store vectors in FAISS
matrix = np.array(chunk_vectors, dtype="float32")
index = faiss.IndexFlatL2(len(chunk_vectors[0]))
index.add(matrix)


In [ ]:
query = "What were the main ideas in the history of art document?"

query_embedding = client.embeddings.create(
    model="text-embedding-3-small",
    input=[query]
).data[0].embedding

In [ ]:
k = 3
D, I = index.search(np.array([query_embedding], dtype="float32"), k)
retrieved_chunks = [chunks[i] for i in I[0]]
retrieved_chunks

In [ ]:
context = "\n\n".join(retrieved_chunks)

prompt = f"""
Answer the question using only the context below.

Context:
{context}

Question:
{query}
"""

In [ ]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You answer questions based only on the provided context."},
        {"role": "user", "content": prompt}
    ]
)

print(response.choices[0].message.content)